# Equilibrium H → process Y: climate-pair transfer audit

Purpose: examine whether climate ranking preserved by a reduced equilibrium representation H is also preserved for the downstream process Y. Pairwise reversal transfer is the primary endpoint; Kendall tau is a secondary aggregate.

**Input provenance:** exact filenames and column names were inspected in Google Drive, folder `process_fidelity_runs/20261115_192400_v1.6.5-climate-cache-key-fix`, on 2026-11-18. No search-by-filename or implicit substitute is used. All outputs are written into a new folder under the same run.

**Eligibility:** IRA lower and upper are independent branch cells; ImCOF has no canonical whole-range H in the completed run. The existing P0 ImCOF lower/upper H is *not* silently joined to its whole-cycle Y. Until `ImCOF_whole_range_process_aligned_H.csv` has been separately reconstructed and regression-verified, ImCOF H→Y is explicitly NOT EVALUABLE; its Y-only representation fidelity is still computed.

**Comparison:** for each material × operating mode × branch, use the *same exact finite climate intersection* across H_single, H_tensor, Y_single, Y_tensor. All strict pair decisions exclude ties, while tie counts are reported. Pairwise results are descriptive; climate pairs are not independent samples.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, itertools, json
from scipy.stats import kendalltau, spearmanr

RUN = Path('/content/drive/MyDrive/process_fidelity_runs/20261115_192400_v1.6.5-climate-cache-key-fix')
if not RUN.exists():
    raise RuntimeError('Canonical run is not accessible. Mount Google Drive from Colab Files panel first; then rerun.')
OUT = RUN/'HY_process_ranking_transfer_audit_v1'
OUT.mkdir(exist_ok=True)

FILES = {
 'p0': 'equilibrium_representation_regression.csv',
 'ira_hy': 'H_Y_climate_scores_aligned.csv',
 'ira_env': 'IRA_environmental_loading_climate_scores.csv',
 'ira_active': 'IRA_active_cycle_climate_scores.csv',
 'im_env': 'ImCOF_environmental_thermal_climate_scores.csv',
 'im_active': 'ImCOF_active_thermal_climate_scores.csv',
 'y_reversals': 'pairwise_climate_reversals.csv',
}
for alias,name in FILES.items():
    p=RUN/name
    if not p.is_file(): raise FileNotFoundError(f'Verified Drive file is missing: {p}')

EXPECTED={
 'p0': ['rep','key','reproduced','reference','material','abs_err','pass','check'],
 'ira_hy':['climate','H','Y','material','controller','representation','branch','temporal_mode','variant','is_primary','n_H_evaluable_climates','n_joint_evaluable_climates'],
 'ira_env':['material','controller','variant','branch','temporal_mode','representation','level','climate','supported_state_fraction','Y','endpoint','propagator'],
 'ira_active':['branch','temporal_mode','climate','Y_active','representation','regen_kernel','bridge_endpoint','material','controller','level','variant'],
 'im_env':['material','controller','thermal_mode','representation','level','thermal_variant','theta','climate','execution_tier','n_cycles','n_supported_cycles','supported_cycle_fraction','Y_env','Y_env_supported_sum','mean_q_rich','mean_F_R','endpoint_definition','interpretation'],
 'im_active':['material','controller','representation','level','window','ads_condition','climate','n_cycles','n_supported_cycles','supported_cycle_fraction','Y_active','Y_active_supported_sum','Y_active_pooled_kernel','Y_active_pooled_kernel_supported_sum','mean_q_rich','mean_q_lean','frac_cycles_outside_support','mean_kernel_distance','endpoint_definition'],
 'y_reversals':['climate_i','climate_j','sign_full','sign_rep','kind','Y_full_i','Y_full_j','Y_rep_i','Y_rep_j','material','controller','representation','variant','branch','temporal_mode','endpoint','n_evaluable_climates','evaluable_climates','window','ads_condition','thermal_variant','theta']
}
RAW={k:pd.read_csv(RUN/v) for k,v in FILES.items()}
SCHEMA=[]
for k,df in RAW.items():
    exp=EXPECTED[k]
    missing=set(exp)-set(df.columns)
    SCHEMA.append({'file':FILES[k],'n_rows':len(df),'columns_actual':'|'.join(df.columns),
                   'columns_expected':'|'.join(exp),'missing_required':'|'.join(sorted(missing))})
    assert not missing,(k,missing)
SCHEMA=pd.DataFrame(SCHEMA)
SCHEMA.to_csv(OUT/'input_schema_audit.csv',index=False)
print(SCHEMA[['file','n_rows','missing_required']].to_string(index=False))

In [ ]:
# Fixed filters; no implicit first-row selection.
p0=RAW['p0'].copy()
p0=p0[(p0.check=='P0_regression') &
      p0['pass'].astype(str).str.lower().isin(['true','1','1.0']) &
      p0.rep.isin(['single','tensor'])].copy()
pk=p0.key.astype(str).str.split('|',expand=True)
assert pk.shape[1]==3, 'P0 key is not branch|temporal_mode|climate'
p0[['branch','temporal_mode','climate']]=pk[[0,1,2]].to_numpy()
p0['H']=pd.to_numeric(p0.reference,errors='coerce')
assert not p0.duplicated(['material','branch','temporal_mode','climate','rep']).any()
assert np.allclose(p0.H,pd.to_numeric(p0.reproduced),equal_nan=True,rtol=1e-7,atol=1e-9)

# Directly reconstructed IRA H–Y table with keys from finished run.
hy=RAW['ira_hy'].copy()
hy=hy[(hy.material=='IRA-900-C') &
      hy.is_primary.astype(str).str.lower().isin(['true','1','1.0']) &
      (hy.temporal_mode=='subsample_1h') &
      hy.representation.isin(['single','tensor']) &
      hy.branch.isin(['lower','upper'])].copy()
assert not hy.duplicated(['controller','representation','branch','climate']).any()

# Regression check canonical P0 H against transfer-table H.
check=hy.merge(p0[p0.material=='IRA-900-C'][['rep','branch','temporal_mode','climate','H']],
               left_on=['representation','branch','temporal_mode','climate'],
               right_on=['rep','branch','temporal_mode','climate'],how='left',validate='many_to_one',suffixes=('','_P0'))
assert check.H_P0.notna().all(),'IRA transfer H has no P0 counterpart'
assert np.allclose(check.H,check.H_P0,rtol=1e-7,atol=1e-8), 'IRA HY H not identical to canonical P0'

# Regression check Y against exact raw score source (including branch/controller).
env=RAW['ira_env']
env=env[(env.material=='IRA-900-C')&(env.controller=='environmental_moisture')&(env.temporal_mode=='subsample_1h')]
act=RAW['ira_active']
act=act[(act.material=='IRA-900-C')&(act.controller=='active_moisture')&(act.temporal_mode=='subsample_1h')& (act.regen_kernel=='primary')]
raw_ira=pd.concat([
 env[['controller','representation','branch','climate','Y']],
 act[['controller','representation','branch','climate','Y_active']].rename(columns={'Y_active':'Y'})],ignore_index=True)
assert not raw_ira.duplicated(['controller','representation','branch','climate']).any(),'Raw IRA scores are nonunique'
g=hy.merge(raw_ira,on=['controller','representation','branch','climate'],how='left',validate='one_to_one',suffixes=('','_raw'))
assert g.Y_raw.notna().all()
assert np.allclose(g.Y,g.Y_raw,rtol=1e-9,atol=1e-10),'IRA transfer Y mismatches process scores'
print('IRA P0 + process regression: PASS; records:',len(g))

In [ ]:
# ImCOF whole-cycle Y — no H assigned until whole-range file is validated.
ia=RAW['im_active']
ia=ia[(ia.material=='ImCOF-TAEA')&(ia.controller=='active_thermal_80C')&
      (ia.window=='primary')&(ia.ads_condition=='end_of_window') & ia.representation.isin(['single','tensor'])].copy()
ie=RAW['im_env']
ie=ie[(ie.material=='ImCOF-TAEA')&(ie.controller=='environmental_thermal_PADAC')&
      (ie.thermal_variant=='primary')&(ie.theta=='PADAC_13X_FIELD_REALISATION') &
      (ie.execution_tier=='PRIMARY')&ie.representation.isin(['single','tensor'])].copy()
for z in [ia,ie]:
    assert not z.duplicated(['climate','representation']).any()
imy=pd.concat([
 ia[['climate','representation','Y_active','supported_cycle_fraction','n_supported_cycles']]
 .rename(columns={'Y_active':'Y'}).assign(controller='active_thermal_80C'),
 ie[['climate','representation','Y_env','supported_cycle_fraction','n_supported_cycles']]
 .rename(columns={'Y_env':'Y'}).assign(controller='environmental_thermal_PADAC')],ignore_index=True)
imy.to_csv(OUT/'ImCOF_verified_primary_Y.csv',index=False)
print(imy.groupby(['controller','representation']).agg(n=('Y','count'),min_cov=('supported_cycle_fraction','min')).to_string())

## Optional ImCOF gate

The completed v1.6.5 run contains P0 **branch** H, not **whole-range** H. Branch H cannot be relabelled as whole H. Only accept a dedicated reconstructed file with columns `climate,representation,H_whole` and the separate `whole_cycle_Y_reconstruction_gate.csv` indicating numeric reproduction of both active/environmental Y. Its absence is an explicit NOT EVALUABLE result, not a substituted estimate.

In [ ]:
# Only consume an earlier standalone reconstruction if both hard-gate evidence files exist.
DUAL=RUN/'ImCOF_dual_alignment_wholeH_branchY'
whole_path=DUAL/'ImCOF_whole_range_process_aligned_H.csv'
gate_path=DUAL/'whole_cycle_Y_reconstruction_gate.csv'
IM_H_STATUS='NOT_EVALUABLE_NO_VERIFIED_WHOLE_RANGE_H'
imh=None
if whole_path.exists() and gate_path.exists():
    wh=pd.read_csv(whole_path)
    gg=pd.read_csv(gate_path)
    assert {'climate','representation','H_whole'}<=set(wh.columns)
    assert {'Y_saved','Y_reconstructed','operating_mode','representation','climate'}<=set(gg.columns)
    assert len(gg)>=100, 'Incomplete process reconstruction gate'
    pres=np.isfinite(gg.Y_saved)==np.isfinite(gg.Y_reconstructed)
    err=(pd.to_numeric(gg.Y_saved,errors='coerce')-pd.to_numeric(gg.Y_reconstructed,errors='coerce')).abs()
    assert pres.all() and err.dropna().max()<1e-8, 'ImCOF whole-cycle Y regression failed'
    assert not wh.duplicated(['climate','representation']).any()
    imh=wh[['climate','representation','H_whole']].rename(columns={'H_whole':'H'})
    IM_H_STATUS='VERIFIED_WHOLE_RANGE_H_REGRESSION_GATE_PASSED'
print('ImCOF H status:',IM_H_STATUS)

In [ ]:
def tau(x,y):
    if len(x)<3 or len(set(x))<2 or len(set(y))<2:return np.nan
    return float(kendalltau(x,y,variant='b').statistic)

def rho(x,y):
    if len(x)<3 or len(set(x))<2 or len(set(y))<2:return np.nan
    return float(spearmanr(x,y).statistic)

def paired_table(df,keys,val):
    assert not df.duplicated(keys+['representation']).any(), f'Duplicate keys for {val}'
    p=df.pivot(index=keys,columns='representation',values=val)
    assert {'single','tensor'}<=set(p.columns)
    return p[['single','tensor']].rename(columns={'single':val+'_single','tensor':val+'_tensor'}).reset_index()

def build_pairs(df,mode,material,branch):
    # One source table per mode+material+branch with 4 numeric columns.
    z=df[['climate','H_single','H_tensor','Y_single','Y_tensor']].copy()
    for c in z.columns[1:]:z[c]=pd.to_numeric(z[c],errors='coerce')
    assert not z.climate.duplicated().any()
    n_input=len(z)
    z=z.replace([np.inf,-np.inf],np.nan).dropna()
    rec=[]
    for a,b in itertools.combinations(z.sort_values('climate').to_dict('records'),2):
        d={v:float(a[v]-b[v]) for v in ['H_single','H_tensor','Y_single','Y_tensor']}
        hs,hf,ys,yf=(np.sign(d[t]) for t in ['H_single','H_tensor','Y_single','Y_tensor'])
        h_tie=hs==0 or hf==0; y_tie=ys==0 or yf==0
        h_rev=(hs!=hf) if not h_tie else np.nan
        y_rev=(ys!=yf) if not y_tie else np.nan
        rec.append({'material':material,'operating_mode':mode,'branch':branch,
                    'climate_i':a['climate'],'climate_j':b['climate'],
                    **{'delta_'+k:d[k] for k in d},
                    'H_tie':h_tie,'Y_tie':y_tie,'H_reversed':h_rev,'Y_reversed':y_rev,
                    'joint_strict_eligible':not(h_tie or y_tie)})
    pp=pd.DataFrame(rec)
    summary={'material':material,'operating_mode':mode,'branch':branch,
             'n_input_climates':n_input,'n_common_finite_climates':len(z),
             'n_pairs':len(pp),'n_H_tie_pairs':int(pp.H_tie.sum()) if len(pp) else 0,
             'n_Y_tie_pairs':int(pp.Y_tie.sum()) if len(pp) else 0,
             'tau_H_single_tensor':tau(z.H_single,z.H_tensor),
             'tau_Y_single_tensor':tau(z.Y_single,z.Y_tensor),
             'rho_H_single_tensor':rho(z.H_single,z.H_tensor),
             'rho_Y_single_tensor':rho(z.Y_single,z.Y_tensor)}
    eligible=pp[pp.joint_strict_eligible] if len(pp) else pp
    n=len(eligible)
    if n:
        hh=eligible.H_reversed.astype(bool); yy=eligible.Y_reversed.astype(bool)
        n00=int((~hh&~yy).sum()); n01=int((~hh&yy).sum())
        n10=int((hh&~yy).sum()); n11=int((hh&yy).sum())
        summary.update(n_joint_strict_pairs=n,n_Hsafe_Ysafe=n00,n_Hsafe_Yrev=n01,
                       n_Hrev_Ysafe=n10,n_Hrev_Yrev=n11,
                       P_Yrev_given_Hsafe=n01/(n00+n01) if n00+n01 else np.nan,
                       P_Yrev_given_Hrev=n11/(n10+n11) if n10+n11 else np.nan,
                       P_Yrev_base=(n01+n11)/n,
                       Hsafe_rate=(n00+n01)/n)
    else:summary.update(n_joint_strict_pairs=0)
    return pp,summary,z

In [ ]:
# IRA: compare H/Y on the exact common climate intersection for each branch and mode.
ALL_PAIRS=[]; SUMMARY=[]; SCORES=[]
for (mode,branch),g in hy.groupby(['controller','branch']):
    hp=paired_table(g,['climate'],'H')
    yp=paired_table(g,['climate'],'Y')
    z=hp.merge(yp,on='climate',validate='one_to_one')
    pp,s,zz=build_pairs(z,mode,'IRA-900-C',branch)
    ALL_PAIRS.append(pp); SUMMARY.append(s)
    SCORES.append(zz.assign(material='IRA-900-C',operating_mode=mode,branch=branch))

# ImCOF: whole-range H can enter only if explicit data and regression gate exist.
if imh is not None:
    hp=paired_table(imh,['climate'],'H')
    for mode,g in imy.groupby('controller'):
        yp=paired_table(g,['climate'],'Y')
        z=hp.merge(yp,on='climate',validate='one_to_one')
        pp,s,zz=build_pairs(z,mode,'ImCOF-TAEA','whole_range')
        ALL_PAIRS.append(pp); SUMMARY.append(s)
        SCORES.append(zz.assign(material='ImCOF-TAEA',operating_mode=mode,branch='whole_range'))

PAIRS=pd.concat(ALL_PAIRS,ignore_index=True)
SUMMARY=pd.DataFrame(SUMMARY)
SCORES=pd.concat(SCORES,ignore_index=True)
PAIRS.to_csv(OUT/'HY_transfer_pairwise.csv',index=False)
SUMMARY.to_csv(OUT/'HY_transfer_summary.csv',index=False)
SCORES.to_csv(OUT/'HY_transfer_aligned_scores.csv',index=False)
display(SUMMARY)

In [ ]:
# Y-only ImCOF ranking fidelity is evaluated without pretending it is H→Y transfer.
Y_ONLY=[]
for mode,g in imy.groupby('controller'):
    z=paired_table(g,['climate'],'Y').dropna()
    Y_ONLY.append({'material':'ImCOF-TAEA','operating_mode':mode,
                   'n':len(z),'tau_Y_single_tensor':tau(z.Y_single,z.Y_tensor),
                   'rho_Y_single_tensor':rho(z.Y_single,z.Y_tensor)})
Y_ONLY=pd.DataFrame(Y_ONLY)
Y_ONLY.to_csv(OUT/'ImCOF_Y_only_fidelity.csv',index=False)
display(Y_ONLY)

In [ ]:
# Reproduce any raw Y reversal entries that are eligible for the selected IRA cells.
# The stored file records reversal-only pairs, not the full pair universe.
old=RAW['y_reversals']
REV_CHECK=[]
for keys,g in PAIRS.groupby(['material','operating_mode','branch']):
    material,mode,branch=keys
    if material!='IRA-900-C':continue
    oe=old[(old.material==material)&(old.controller==mode)&(old.branch==branch)&
           (old.representation=='single')&(old.kind=='STRICT_REVERSAL')]
    calc=set(tuple(sorted([r.climate_i,r.climate_j])) for r in g[g.Y_reversed==True].itertuples())
    observed=set(tuple(sorted([r.climate_i,r.climate_j])) for r in oe.itertuples())
    REV_CHECK.append({'material':material,'operating_mode':mode,'branch':branch,
                      'n_calculated_Y_reversal_pairs':len(calc),
                      'n_stored_Y_reversal_pairs':len(observed),
                      'stored_minus_calculated':len(observed-calc),
                      'calculated_minus_stored':len(calc-observed)})
REV_CHECK=pd.DataFrame(REV_CHECK)
REV_CHECK.to_csv(OUT/'stored_Y_reversal_regression.csv',index=False)
display(REV_CHECK)
if (REV_CHECK.stored_minus_calculated>0).any():
    print('WARNING: stored reversal table has pairs outside current H/Y common support; inspect rather than assuming regression equality.')

## Robustness: support, margin, climate leverage

Primary pairs use the exact finite intersection. A margin filter is diagnostic only: mark the 5% relative-H reference margin used in the manuscript and report dependence, without silently applying a new process-Y threshold. Leave-one-climate-out recomputes transfer fractions with dependence preserved. No pairwise binomial p-values are calculated.

In [ ]:
ROB=[]; LO=[]
for keys,g in PAIRS.groupby(['material','operating_mode','branch']):
    mat,mode,branch=keys
    sc=SCORES[(SCORES.material==mat)&(SCORES.operating_mode==mode)&(SCORES.branch==branch)]
    for removed in [None]+sorted(sc.climate.tolist()):
        p=g if removed is None else g[(g.climate_i!=removed)&(g.climate_j!=removed)]
        eligible=p[p.joint_strict_eligible]
        if len(eligible):
            h=eligible.H_reversed.astype(bool); y=eligible.Y_reversed.astype(bool)
            mask=(~h)
            bad=float(y[mask].mean()) if mask.any() else np.nan
        else:bad=np.nan
        LO.append({'material':mat,'operating_mode':mode,'branch':branch,
                   'left_out_climate':removed if removed is not None else '__NONE__',
                   'n_pairs':len(p),'P_Yrev_given_Hsafe':bad})
    # 5% relative-H margin diagnostic, tied to tensor H (same score basis as H comparison)
    z=sc.set_index('climate')
    ratios=[]
    for rr in g.itertuples(index=False):
        hi,hj=float(z.loc[rr.climate_i,'H_tensor']),float(z.loc[rr.climate_j,'H_tensor'])
        hm=(abs(hi)+abs(hj))/2
        ratios.append(abs(hi-hj)/hm if hm>0 else np.nan)
    g=g.copy();g['H_tensor_relative_margin']=ratios
    for th in [0.0,0.05,0.10]:
        a=g[(g.H_tensor_relative_margin>=th)&g.joint_strict_eligible]
        if len(a):
            h=a.H_reversed.astype(bool); y=a.Y_reversed.astype(bool)
            n00=int((~h&~y).sum());n01=int((~h&y).sum());n10=int((h&~y).sum());n11=int((h&y).sum())
        else:n00=n01=n10=n11=0
        ROB.append({'material':mat,'operating_mode':mode,'branch':branch,'min_rel_H_tensor_margin':th,
                    'n_eligible_pairs':len(a),'n_Hsafe_Yrev':n01,
                    'P_Yrev_given_Hsafe':n01/(n00+n01) if n00+n01 else np.nan,
                    'P_Yrev_given_Hrev':n11/(n10+n11) if n10+n11 else np.nan,
                    'P_Yrev_base':(n01+n11)/len(a) if len(a) else np.nan})
ROB=pd.DataFrame(ROB); LO=pd.DataFrame(LO)
ROB.to_csv(OUT/'HY_transfer_margin_sensitivity.csv',index=False)
LO.to_csv(OUT/'HY_transfer_leave_one_climate_out.csv',index=False)
display(ROB)

In [ ]:
# Climate-level coverage sensitivity can be implemented only for ImCOF when H is verified.
COVERAGE=[]
if imh is not None:
    for mode,g in imy.groupby('controller'):
        cov=g[g.representation=='tensor'][['climate','supported_cycle_fraction']]
        h=paired_table(imh,['climate'],'H')
        y=paired_table(g,['climate'],'Y')
        z=h.merge(y,on='climate').merge(cov,on='climate')
        for threshold in [0.0,0.01,0.05,0.10,0.20]:
            zz=z[z.supported_cycle_fraction>=threshold]
            _,s,_=build_pairs(zz,mode,'ImCOF-TAEA','whole_range')
            s['min_supported_cycle_fraction']=threshold
            COVERAGE.append(s)
COVERAGE=pd.DataFrame(COVERAGE)
if len(COVERAGE): COVERAGE.to_csv(OUT/'ImCOF_transfer_coverage_sensitivity.csv',index=False)

In [ ]:
status=pd.DataFrame([
 {'analysis':'IRA branch P0 H / IRA process Y transfer','status':'EXECUTED_WITH_P0_AND_Y_REGRESSION','evidence':str(OUT/'HY_transfer_summary.csv')},
 {'analysis':'ImCOF whole-range H / whole-cycle Y transfer','status':IM_H_STATUS,'evidence':str(whole_path) if imh is not None else 'P0 lower/upper H cannot substitute whole-range H'},
 {'analysis':'ImCOF whole-cycle Y single/tensor fidelity','status':'EXECUTED_Y_ONLY','evidence':str(OUT/'ImCOF_Y_only_fidelity.csv')},
 {'analysis':'ImCOF conditional branch Y vs P0 branch H','status':'EXCLUDED_FROM_PRIMARY','evidence':'Different endpoint; not additive whole-cycle decomposition'},
 {'analysis':'ImCOF independent process experimental validation','status':'NOT_AVAILABLE','evidence':'PADAC 13X temperature forcing is not measured ImCOF reactor temperature'}
])
status.to_csv(OUT/'execution_and_claim_status.csv',index=False)
with open(OUT/'README_claim_boundaries.md','w') as f:
    f.write('# H-to-Y transfer analysis\n\nIRA executed on exact P0 and raw Y with regression gates.\n\n'
            +f'ImCOF H-to-Y status: {IM_H_STATUS}.\n\n'
            +'Never treat H–Y correspondence as independent reactor validation. Climate pairs are dependent. '
            +'H safe is a pair-ordering test, not a measured process-safety guarantee.\n')
print('Output folder:',OUT)
display(status)